In [1]:
import pandas as pd
import numpy as np
import missingno as msno

In [2]:
df = pd.read_csv("../data/StudentsPerformance.csv") #Importación inicial del archivo

# Análisis de rendimiento académico de estudiantes - Big Data IDIA 222

Con el fin de probar la reproductibilidad de archivos de código python mediante la aplicación de conocimientos previos sobre el manejo de repositorios y clonación de entornos para su ejecución, se realizará un trabajo de análisis y procesamiento de datos con determinados parámetros (sobre librerías, enfáticamente) para su posterior reproducción en una máquina tercera.

In [3]:
#Despliegue inicial de CSV para verificar importación
df

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75
...,...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,88,99,95
996,male,group C,high school,free/reduced,none,62,55,55
997,female,group C,high school,free/reduced,completed,59,71,65
998,female,group D,some college,standard,completed,68,78,77


## Exploración inicial

Lo primero siempre se tratará de conocer la base de datos con la que estamos trabajando, en este proceso se examinan detalles como:
- número de registros
- Número de columnas
- Número de variables
- Tipos de datos
- Valores nulos (o faltantes)
- Duplicados (ya sea registros o columnas)
- Estadística descrpitiva

In [4]:
print("Exploración inicial")
print("Número de registros: ", df.shape[0]*df.shape[1])
print("Número de columnas: ", df.shape[1])
print("Número de filas: ", df.shape[0])

Exploración inicial
Número de registros:  8000
Número de columnas:  8
Número de filas:  1000


In [7]:
#Para tipos de datos, esperando encontrar datos categóricos como "str" y numéricos como "int64"/"float64"
df.dtypes 

gender                           str
race/ethnicity                   str
parental level of education      str
lunch                            str
test preparation course          str
math score                     int64
reading score                  int64
writing score                  int64
dtype: object

In [8]:
#Para evaluación de datos nulos, se opta por realizar el análisis por columna y por porcentaje
df.isnull().mean()*100

gender                         0.0
race/ethnicity                 0.0
parental level of education    0.0
lunch                          0.0
test preparation course        0.0
math score                     0.0
reading score                  0.0
writing score                  0.0
dtype: float64

In [19]:
#Para la verificación de duplicados en ambas columnas y filas:
duplicados_filas = df[df.duplicated()]

print("Filas duplicadas:")
duplicados_filas

duplicados_columnas = df.columns[df.columns.duplicated()]

print("Columnas duplicadas:")

if len(duplicados_columnas) > 0:
    display(pd.DataFrame({"Columnas duplicadas": duplicados_columnas}))
else:
    display(pd.DataFrame())

Filas duplicadas:
Columnas duplicadas:


""


In [23]:
#Estadística descriptiva (variables categóricas)
df.describe(include="str")

,gender,race/ethnicity,parental level of education,lunch,test preparation course
count,1000,1000,1000,1000,1000
unique,2,5,6,2,2
top,female,group C,some college,standard,none
freq,518,319,226,645,642


In [24]:
#Estadística descriptiva (datos numéricos)
df.describe()

,math score,reading score,writing score
count,1000.00000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000
std,15.16308,14.600192,15.195657
min,0.00000,17.000000,10.000000
25%,57.00000,59.000000,57.750000
50%,66.00000,70.000000,69.000000
75%,77.00000,79.000000,79.000000
max,100.00000,100.000000,100.000000


De forma general podemos ver que el dataframe inicial con el que trabajamos tiene un buen estado en lo que concierne a duplicados y/o nulos, ya que de ninguno de los dos se encontraron registros que puedan afectar al uso del mismo. Con esto se puede comenzar a hacer evaluaciones e identificación de patrones, aunque la estadística descrpitiva ya revela, cuando menos, los patrones más elementales de comportamiento de los datos.

## Limpieza y preprocesamientos

Como comprobamos en el punto anterior, el dataframe inicial presenta un buen estado de salud considerando que no existen duplicados ni nulos; por otro lado, únicamente se hará una evaluación sobre valores posibles para identificar:
- En variables categóricas: todas las respuestas únicas que hay disponibles
- En variables numéricas: rangos de valores

Esto con el fin de corroborar si no existen datos "extraños" en el dataframe.

In [30]:
#Análisis de valores únicos en variables categóricas
col_cat = ["gender", "race/ethnicity", "parental level of education", "lunch", "test preparation course"]
for col in col_cat:
    print()
    print(f"Columna:  {col}")
    print(f"Total de observaciones: {df[col].count()}")
    tabla = pd.DataFrame({
        "Frecuencia": df[col].value_counts()
    })
    display(tabla)


Columna:  gender
Total de observaciones: 1000


,Frecuencia
gender,
female,518
male,482



Columna:  race/ethnicity
Total de observaciones: 1000


,Frecuencia
race/ethnicity,
group C,319
group D,262
group B,190
group E,140
group A,89



Columna:  parental level of education
Total de observaciones: 1000


,Frecuencia
parental level of education,
some college,226
associate's degree,222
high school,196
some high school,179
bachelor's degree,118
master's degree,59



Columna:  lunch
Total de observaciones: 1000


,Frecuencia
lunch,
standard,645
free/reduced,355



Columna:  test preparation course
Total de observaciones: 1000


,Frecuencia
test preparation course,
none,642
completed,358


In [28]:
#Análisis de rango de valores en variables numéricas.
col_num = ["math score", "reading score", "writing score"]
for col in col_num:
    print()
    print(f"Columna: {col}, mínimo: {df[col].min()}, Máximo: {df[col].max()}")


Columna: math score, mínimo: 0, Máximo: 100

Columna: reading score, mínimo: 17, Máximo: 100

Columna: writing score, mínimo: 10, Máximo: 100


Con estas observaciones podemos descatar si hay o no algun dato irregular o que no pertenezca a las respuestas esperadas del dataframe. A pesar de ello, no se encontró ningún dato irregular, por lo que no es necesario realizar limpieza ni procesos especiales al momento.

## Clasificación de rendimiento por calificación promedio

En este apartado se creará una nueva variable que englobe, de cada registro, el promedio general de las tres materias para tomar de referencia para una clasificación del rendimiento académico del estudiante. 

La ponderación del mismo será dividida en cinco categorías basadas en el rango de calificación final que tenga: 
- Excelente (Excellent): 95-100
- Potencial (Potential): 90-94.9999
- Sobresaliente (Outstanding): 80-89.9999
- Suficiente (Enough): 70-79.9999
- Deficiente (Deficient): <70

Originalmente se planteaba la posibilidad de incluir de referencia el uso de la categoría "test preparation course" ya que este indica si se hizo un curso de preparación previo al examen; se pensaba que, si el alumno tenía una calificación de entre 90 y 100 sin haber hecho el examen, podría considerarse "sobresaliente" al no contar con preparación, sin embargo la idea se descartó al considerarse irrelevante y que siempre existe estudio del que no se tiene constancia por parte del personal docente (cosa que en el márgen de los hechos, es más común). 

In [31]:
# creación de variable con el promedio general de cada alumno
df["Average_Final"] = df[
    ["math score", "reading score", "writing score"]
].mean(axis=1)
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score,Average_Final
0,female,group B,bachelor's degree,standard,none,72,72,74,72.666667
1,female,group C,some college,standard,completed,69,90,88,82.333333
2,female,group B,master's degree,standard,none,90,95,93,92.666667
3,male,group A,associate's degree,free/reduced,none,47,57,44,49.333333
4,male,group C,some college,standard,none,76,78,75,76.333333


## Respuesta a preguntas sobre la información:

### ¿Cuál de las tres áreas tiene el promedio más alto?

Este análisis se puede obtener con un comando ejecutado anteriormente:
~~~
df.describe()
~~~

In [36]:
df.describe()

,math score,reading score,writing score,Average_Final
count,1000.00000,1000.000000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000,67.770667
std,15.16308,14.600192,15.195657,14.257326
min,0.00000,17.000000,10.000000,9.000000
25%,57.00000,59.000000,57.750000,58.333333
50%,66.00000,70.000000,69.000000,68.333333
75%,77.00000,79.000000,79.000000,77.666667
max,100.00000,100.000000,100.000000,100.000000


**R:** El área de lectura (reading) es la que cuenta con el promedio general (de los 1000 alumnos) más alto, con un 69.19, seguido de el área de escritura (writing) con 98.05.

### ¿Los estudiantes que realizaron el curso de preparación presentan mejores resultados?

Este análisis se puede hacer si verificamos el promedio de calificaciones considerando la columna sobre el curso de preparación como referencia; se filtran los registros con el criterio sobre la variable mencionada, de forma que tenemos dos dataframes de menor escala, y de cada uno obtenemos el promedio general a partir de una descripción (`df.describe`) de la columna `Average_Final`.

In [37]:
# Crea un dataframe nuevo y diferente basado en la respuesta de la variable sobre el curso de preparación
df_completed = df[df["test preparation course"] == "completed"]

df_none = df[df["test preparation course"] == "none"]

In [38]:
df_completed.describe()

,math score,reading score,writing score,Average_Final
count,358.000000,358.000000,358.000000,358.000000
mean,69.695531,73.893855,74.418994,72.669460
std,14.444699,13.638384,13.375335,13.036960
min,23.000000,37.000000,36.000000,34.333333
25%,60.000000,65.000000,66.000000,65.000000
50%,69.000000,75.000000,76.000000,73.500000
75%,79.000000,84.000000,83.000000,82.166667
max,100.000000,100.000000,100.000000,100.000000


In [39]:
df_none.describe()

,math score,reading score,writing score,Average_Final
count,642.000000,642.000000,642.000000,642.000000
mean,64.077882,66.534268,64.504673,65.038941
std,15.192376,14.463885,14.999661,14.186707
min,0.000000,17.000000,10.000000,9.000000
25%,54.000000,57.000000,54.000000,55.416667
50%,64.000000,67.000000,65.000000,65.333333
75%,74.750000,76.000000,74.000000,75.000000
max,100.000000,100.000000,100.000000,100.000000


Con este análisis podemos revelar el promedio final de, solamente, los registros que en la variable `test preparation course` tengan por valor ya sea `none` o `completed`.

**R:** En general, los estudiantes que *si* concluyeron el curso tienden a tener mayor calificación, teniendo un promedio general de 72.66, por lo tanto **sí**, tienen mejor rendimiento.

### ¿Qué porcentaje de estudiantes alcanza un promedio de valoración "sobresaliente" (outstanding) o superior?
Este análisis es similar al anterior en el sentido de que establece una condición para realizar un conteo de falsos y verdaderos, esta vez, basado en el valor de `Average_Final`. 

Se estableció previamente que la clase `Outstanding` es para calificaciones de 80 o superiores, por lo que se tomará el criterio numérico ya que reduce la cantidad de condiciones que habría que declarar si se escribe verbalmente. 

Cada condición se guarda en una variable que ayudará a asignar un valor final para impresión; lo que revelará el porcentaje de la ponderación binarizada (más parecido a aprobado/reprobado).

In [41]:
Outstanding = df["Average_Final"] >= 80
Barely = df["Average_Final"] <80

print(f"{Outstanding.mean()*100}% de los estudiantes tiene un rendimiento sobresaliente o mejor")
print(f"{Barely.mean()*100}% de los estudiantes tiene un rendimiento apenas suficiente o deficiente")


19.8% de los estudiantes tiene un rendimiento sobresaliente o mejor
80.2% de los estudiantes tiene un rendimiento apenas suficiente o deficiente


**R:** Podemos ver que un porcentaje relativamente bajo tiene un promedio sobresaliente o superior, siendo de poco menos del `20%` aquellos que cuentan con una calificación final de 80 o mayor.

### ¿Cuál es el grupo con menor promedio y cuál es el que tiene el mayor?
Este análisis vuelve a realizar un filtrado por variable para crear dataframes reducidos con respecto al valor de la misma; en este caso relacionando cada uno con los grupos que hay. Se cuentan con 5 grupos, del A al E, por lo que se crearán las variables condicionadas con respecto a esto. para después realizar una descripción estadística de cada uno.

In [42]:
#Asignación y creación de nuevas variables por grupo de población
df_A = df[df["race/ethnicity"] == "group A"]
df_B = df[df["race/ethnicity"] == "group B"]
df_C = df[df["race/ethnicity"] == "group C"]
df_D = df[df["race/ethnicity"] == "group D"]
df_E = df[df["race/ethnicity"] == "group E"]

In [52]:
df_A.describe()

,math score,reading score,writing score,Average_Final
count,89.000000,89.000000,89.000000,89.000000
mean,61.629213,64.674157,62.674157,62.992509
std,14.523008,15.543762,15.468278,14.444598
min,28.000000,23.000000,19.000000,23.333333
25%,51.000000,53.000000,51.000000,52.000000
50%,61.000000,64.000000,62.000000,61.333333
75%,71.000000,74.000000,73.000000,73.000000
max,100.000000,100.000000,97.000000,96.333333


In [53]:
df_B.describe()

,math score,reading score,writing score,Average_Final
count,190.000000,190.000000,190.000000,190.000000
mean,63.452632,67.352632,65.600000,65.468421
std,15.468191,15.177499,15.625173,14.732133
min,8.000000,24.000000,15.000000,18.333333
25%,54.000000,56.000000,55.250000,56.666667
50%,63.000000,67.000000,67.000000,65.000000
75%,74.000000,79.750000,78.000000,76.833333
max,97.000000,97.000000,96.000000,96.666667


In [54]:
df_C.describe()

,math score,reading score,writing score,Average_Final
count,319.000000,319.000000,319.000000,319.000000
mean,64.463950,69.103448,67.827586,67.131661
std,14.852666,13.997033,14.983378,13.872211
min,0.000000,17.000000,10.000000,9.000000
25%,55.000000,60.000000,57.000000,57.666667
50%,65.000000,71.000000,68.000000,68.333333
75%,74.000000,78.500000,79.000000,77.000000
max,98.000000,100.000000,100.000000,98.666667


In [55]:
df_D.describe()

,math score,reading score,writing score,Average_Final
count,262.000000,262.000000,262.000000,262.000000
mean,67.362595,70.030534,70.145038,69.179389
std,13.769386,13.895306,14.367707,13.252776
min,26.000000,31.000000,32.000000,31.000000
25%,59.000000,60.250000,61.000000,60.333333
50%,69.000000,71.000000,72.000000,70.000000
75%,77.000000,79.000000,80.000000,78.583333
max,100.000000,100.000000,100.000000,99.000000


In [56]:
df_E.describe()

,math score,reading score,writing score,Average_Final
count,140.000000,140.000000,140.000000,140.000000
mean,73.821429,73.028571,71.407143,72.752381
std,15.534259,14.874024,15.113906,14.565016
min,30.000000,26.000000,22.000000,26.000000
25%,64.750000,63.000000,62.000000,64.666667
50%,74.500000,74.000000,72.000000,73.500000
75%,85.000000,84.000000,80.250000,82.416667
max,100.000000,100.000000,100.000000,100.000000


Con este análisis podemos ver los promedios generales de cada grupo, nuevamente basándonos en la variable `Average_Final`

**R:** 
Podemos ver que el __grupo E__ tiene el mayor promedio, con 72.75 de promedio general
Mientras que el __grupo A__ es el que tiene el menor de todos, con 65.99 de promedio general

Esto puede revelar un patrón que podría plantear una posibilidad sobre un criterio de elección de los alumnos de cada grupo, o al menos un patrón muy notorio entre los mismos.